# TPC-H Q14 Join-Update-Join-Update-Join: Codex 1% (Uniform)

Compares: **SNAP, MONO-NR/RR/WR, DUAL-WR, EPOCH-NR/RR/WR** at 1% update (uniform),
with **2 cycles** (join-update-join-update-join).

Graph style matches `new_plot.ipynb` (Paul Tol Bright palette, solid=write, hatch=read).


In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'matplotlib'])
print('done')

done



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
import subprocess, os, sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
SIGMOD = (ROOT / 'benches' / 'sigmod').resolve()
DATA_DIR = SIGMOD / 'tpch_data'
GEN_UPDATES = SIGMOD / 'generate_updates.py'
BIN = ROOT / 'target' / 'release' / 'tpch_q14_join_update_join_codex'

# ===== CONFIG =====
PART_SF      = 1.0
LINEITEM_SF  = 1.0
UPDATE_PCT   = 1.0       # fixed 1%
DIST         = 'uniform'
BUCKET_NUM   = 4096
CYCLES       = 2
WARMUP       = 1
REPEAT       = 10
TRIM         = 2

# All series: (table_type, repair_mode)
SERIES = [
    ('naive', 'nr'),
    ('heap',  'nr'), ('heap',  'rr'), ('heap',  'wr'),
    ('chain', 'wr'),
    ('par',   'nr'), ('par',   'rr'), ('par',   'wr'),
]
# ==================

OUT_CSV = DATA_DIR / 'q14_juj2_1pct_snap_mono_dual_epoch_codex_isolated.csv'

part_file = DATA_DIR / f'part_sf{PART_SF}.tbl'
probe_file = DATA_DIR / f'lineitem_probe_sf{LINEITEM_SF}_1995-09-01_1995-10-01.tbl'

print('ROOT:', ROOT)
print('CSV :', OUT_CSV)



ROOT: /Users/jun/Desktop/dev/mbp16/crustydb/riki/FosterBtree
CSV : /Users/jun/Desktop/dev/mbp16/crustydb/riki/FosterBtree/benches/sigmod/tpch_data/q14_juj2_1pct_snap_mono_dual_epoch_codex_isolated.csv


In [3]:
# ── Build Rust binary ──
print('Building tpch_q14_join_update_join_codex...')
result = subprocess.run(
    ['cargo', 'build', '--release', '--bin', 'tpch_q14_join_update_join_codex'],
    cwd=ROOT, capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('cargo build failed')
print('Build OK')

Building tpch_q14_join_update_join_codex...
Build OK


In [4]:
# ── Ensure update file exists ──
pct_label = f'{UPDATE_PCT:g}'
updates_file = DATA_DIR / f'part_updates_sf{PART_SF}_{pct_label}pct_{DIST}.tbl'

if not updates_file.exists():
    print(f'Generating updates: {updates_file.name}')
    subprocess.run([
        sys.executable, str(GEN_UPDATES),
        str(part_file), str(UPDATE_PCT), str(PART_SF),
        '--output-dir', str(DATA_DIR),
    ], check=True, capture_output=True)
else:
    print(f'Updates file exists: {updates_file.name}')

part_rows = sum(1 for _ in open(part_file))
probe_rows = sum(1 for _ in open(probe_file))
update_rows = sum(1 for _ in open(updates_file))
print(f'PART: {part_rows:,}  PROBE: {probe_rows:,}  UPDATES: {update_rows:,}')

Updates file exists: part_updates_sf1.0_1pct_uniform.tbl
PART: 200,000  PROBE: 75,983  UPDATES: 2,000


In [5]:
# ── Run benchmark for all series ──
if OUT_CSV.exists():
    OUT_CSV.unlink()
    print('Removed old CSV')

total_runs = len(SERIES)
for i, (ttype, rmode) in enumerate(SERIES):
    print(f'[{i + 1}/{total_runs}] table={ttype} repair={rmode}')
    cmd = [
        str(BIN),
        '--part-file', str(part_file),
        '--lineitem-file', str(probe_file),
        '--updates-file', str(updates_file),
        '--table-type', ttype,
        '--repair-mode', rmode,
        '--bucket-num', str(BUCKET_NUM),
        '--cycles', str(CYCLES),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--update-pct', pct_label,
        '--distribution', DIST,
        '--output-csv', str(OUT_CSV),
    ]
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  FAILED: {r.stderr[:400]}')
    else:
        for line in r.stdout.strip().split('\\n'):
            if 'total_ms' in line or 'trimmed' in line:
                print(f'  {line.strip()}')

print(f'\nDone. Results -> {OUT_CSV}')



[1/8] table=naive repair=nr
  Loaded: part_rows=200000, probe_rows=75983, update_ops=2000
warmup 1/1 done
  run 1/10: j1_alloc=0.001, j1_insert=176.918, j1_build=176.918, total=754.003 j1_probe=36.468 | j2_probe=85.995 | j3_probe=84.573 | updates=[u1=0.000 u2=0.000]
  run 2/10: j1_alloc=0.000, j1_insert=163.317, j1_build=163.317, total=776.898 j1_probe=35.705 | j2_probe=82.949 | j3_probe=79.684 | updates=[u1=0.000 u2=0.000]
  run 3/10: j1_alloc=0.000, j1_insert=174.116, j1_build=174.117, total=823.870 j1_probe=36.164 | j2_probe=83.915 | j3_probe=80.650 | updates=[u1=0.000 u2=0.000]
  run 4/10: j1_alloc=0.001, j1_insert=201.311, j1_build=201.311, total=787.515 j1_probe=35.172 | j2_probe=81.793 | j3_probe=82.045 | updates=[u1=0.000 u2=0.000]
  run 5/10: j1_alloc=0.000, j1_insert=175.049, j1_build=175.050, total=774.016 j1_probe=36.818 | j2_probe=86.477 | j3_probe=82.719 | updates=[u1=0.000 u2=0.000]
  run 6/10: j1_alloc=0.001, j1_insert=157.937, j1_build=157.938, total=744.133 j1_probe=3

In [6]:
# ── Load results ──
import re

df = pd.read_csv(OUT_CSV)

# Keep old CSV compatibility (legacy columns)
if ('update_ms' in df.columns) and ('update1_ms' not in df.columns):
    # old format: update_ms represented single update phase
    df['update1_ms'] = pd.to_numeric(df['update_ms'], errors='coerce').fillna(0.0)

if ('join1_build_ms' in df.columns) and ('join1_alloc_ms' not in df.columns) and ('join1_insert_ms' not in df.columns):
    # old format: first build was not split
    df['join1_alloc_ms'] = 0.0
    df['join1_insert_ms'] = pd.to_numeric(df['join1_build_ms'], errors='coerce').fillna(0.0)

update_cols = sorted(
    [c for c in df.columns if re.fullmatch(r'update\d+_ms', c)],
    key=lambda c: int(c[len('update'):-3])
)
join_build_cols = sorted(
    [c for c in df.columns if re.fullmatch(r'join\d+_build_ms', c)],
    key=lambda c: int(c[4:-8])
)
join_probe_cols = sorted(
    [c for c in df.columns if re.fullmatch(r'join\d+_probe_ms', c)],
    key=lambda c: int(c[4:-8])
)

required_cols = ['join1_alloc_ms', 'join1_insert_ms', 'join1_build_ms', 'join1_probe_ms', 'update_pct', 'distribution', 'table_type', 'repair_mode']
for c in required_cols:
    if c not in df.columns:
        df[c] = 0.0

for c in update_cols + join_build_cols + join_probe_cols:
    df[c] = pd.to_numeric(df.get(c, 0.0), errors='coerce').fillna(0.0)

for c in required_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0) if c in {'update_pct'} else df[c]

table_map = {'Heap': 'heap', 'Chain': 'chain', 'Par': 'par', 'Naive': 'naive'}
repair_map = {'Nr': 'NR', 'Rr': 'RR', 'Wr': 'WR'}
df['table'] = df['table_type'].map(table_map).fillna(df['table_type'])
df['repair'] = df['repair_mode'].map(repair_map).fillna(df['repair_mode'])
df.loc[df['table'] == 'naive', 'repair'] = ''

display_cols = ['table', 'repair', 'join1_alloc_ms', 'join1_insert_ms', 'join1_probe_ms', 'join1_build_ms']
display_cols += update_cols
display_cols += join_build_cols
display_cols += join_probe_cols

for col in ['join1_alloc_ms', 'join1_insert_ms', 'join1_build_ms', 'join1_probe_ms']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
for col in update_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
for col in join_build_cols + join_probe_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

display(df[display_cols])



ValueError: invalid literal for int() with base 10: '1_'

In [ ]:
# ── Grouped stacked bar chart (new_plot.ipynb style) ──

import re

# Paul Tol "Bright" palette
tol = {
    "blue":   "#4477AA",
    "cyan":   "#66CCEE",
    "green":  "#228833",
    "yellow": "#CCBB44",
    "red":    "#EE6677",
    "purple": "#AA3377",
    "grey":   "#BBBBBB",
}


def find_max_phase(df):
    join_nums = []
    update_nums = []
    for c in df.columns:
        m = re.fullmatch(r'join(\d+)_probe_ms', c)
        if m:
            join_nums.append(int(m.group(1)))
        m = re.fullmatch(r'update(\d+)_ms', c)
        if m:
            update_nums.append(int(m.group(1)))
    return max(join_nums or [1]), max(update_nums or [0])

join_count, update_count = find_max_phase(df)

# Build phases dynamically: (csv_col, display_name, color, is_write)
phases = [
    ('join1_alloc_ms', 'J1 Alloc', tol['blue'], True),
    ('join1_insert_ms', 'J1 Insert', tol['cyan'], True),
    ('join1_probe_ms', 'J1 Probe', tol['red'], False),
]

for u in range(1, update_count + 1):
    phases.append((f'update{u}_ms', f'Update {u}', tol['yellow'], True))
    j = u + 1
    phases.append((f'join{j}_build_ms', f'J{j} Build', tol['purple'], True))
    phases.append((f'join{j}_probe_ms', f'J{j} Probe', tol['green'], False))

hatch_map = {f'J{i} Probe': '///' for i in range(1, join_count + 1)}

# Table/repair ordering
table_order = ['naive', 'heap', 'chain', 'par']
table_display = {'naive': 'SNAP', 'heap': 'MONO', 'chain': 'DUAL', 'par': 'EPOCH'}
repair_order = {
    'naive': [''],
    'heap': ['NR', 'RR', 'WR'],
    'chain': ['WR'],
    'par': ['NR', 'RR', 'WR'],
}

# Font
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Nimbus Roman No9 L', 'DejaVu Serif']
plt.rcParams['hatch.linewidth'] = 0.9

# Build groups
bar_w = 0.58
gap = 0.36
bar_x, minor_labels, major_centers, major_labels, combos = [], [], [], [], []
x = 0.0
for t in table_order:
    subs = repair_order[t]
    start = x
    for r in subs:
        bar_x.append(x)
        minor_labels.append(r)
        combos.append((t, r))
        x += 0.78
    end = x - 0.78
    major_centers.append((start + end) / 2.0)
    major_labels.append(table_display[t])
    x += gap

fig, ax = plt.subplots(figsize=(8.4, 6.0))
fig.subplots_adjust(bottom=0.22)

for i, (t, r) in enumerate(combos):
    row = df[(df['table'] == t) & (df['repair'] == r)]
    if row.empty:
        continue
    row = row.iloc[0]
    bottom = 0.0
    for col, lbl, color, is_write in phases:
        v = float(row.get(col, 0.0))
        if v <= 0:
            continue
        if is_write:
            ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                   color=color, edgecolor='black', linewidth=0.4, zorder=2)
        else:
            ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                   color='white', edgecolor='black', linewidth=0.4, zorder=2)
            ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                   color='none', edgecolor=color, linewidth=0.9,
                   hatch=hatch_map.get(lbl, '///'), zorder=3)
        bottom += v

# Two-tier x-axis labels
ax.set_xticks([])
for xi, lab in zip(bar_x, minor_labels):
    ax.text(xi, -0.028, lab, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=10, clip_on=False)
for xc, lab in zip(major_centers, major_labels):
    ax.text(xc, -0.072, lab, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=12, clip_on=False)

for k in range(1, len(major_centers)):
    prev_end = max(xx for xx, (tt, _) in zip(bar_x, combos) if tt == table_order[k - 1])
    next_start = min(xx for xx, (tt, _) in zip(bar_x, combos) if tt == table_order[k])
    ax.axvline((prev_end + next_start) / 2, linestyle=':', linewidth=0.6, alpha=0.3)

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax.set_ylabel('Duration (ms)')
ax.set_title('TPC-H Q14 J-U-J-U-J (1% update, uniform)')

# Legend (reversed order, matching new_plot style)
handles, labels_leg = [], []
for col, lbl, color, is_write in reversed(phases):
    if is_write:
        patch = Patch(facecolor=color, edgecolor='black', linewidth=0.4)
    else:
        patch = Patch(facecolor='white', edgecolor=color, hatch=hatch_map.get(lbl, '///'), linewidth=0.9)
    handles.append(patch)
    labels_leg.append(lbl)

ax.legend(handles, labels_leg, title='Operations',
          loc='upper right', framealpha=0.9, fontsize=8.5, title_fontsize=8.5)

plt.tight_layout()
plt.savefig(str(DATA_DIR / 'q14_juj2_1pct_snap_mono_dual_epoch_codex_isolated.pdf'), format='pdf')
plt.show()
print('Saved PDF')
